# Faruq-v3 — AF2_ORIENT + CMC0 seed42

Eksperimen komposisi eksploratif yang sudah diperbaiki: **AF2_ORIENT** sebagai frontend dan **CMC0** hanya pada jalur klasifikasi. Candidate dan frozen parent memakai checkpoint D0 seed42 yang sama. Static audit wajib PASS sebelum training. Tidak ada akses test dan tidak ada tuning setelah seed42.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-orient-cmc0-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
for name in list(sys.modules):
    if name == 'coffee_detector' or name.startswith('coffee_detector.'):
        sys.modules.pop(name, None)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
import ultralytics
assert ultralytics.__version__ == '8.4.96', ultralytics.__version__
print('GPU:', torch.cuda.get_device_name(0))
print('Branch:', BRANCH)


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-af2-isolated-seed42-v1/val_reports/AF2_ORIENT_seed42_result.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
AF2_ORIENT_PARENT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-af2-isolated-seed42-v1/val_reports/AF2_ORIENT_seed42_result.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED = DATA_ROOT/'faruq_grouped_summary.json'
if not GROUPED.is_file():
    if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content', filter='data')
assert (DATA_ROOT/'data.yaml').is_file()
assert GROUPED.is_file()
assert not (DATA_ROOT/'test').exists(), 'STOP: test tidak boleh tersedia.'
OUTPUT = PROJECT_ROOT/'experiments/faruq-v3-af2-orient-cmc0-seed42-v2'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('D0:', D0)
print('AF2_ORIENT parent:', AF2_ORIENT_PARENT)
print('OUTPUT:', OUTPUT)


## 1. Static audit

Membuktikan CMC0 zero-gate identik secara kondisional terhadap AF2_ORIENT, serta bahwa saat gate diaktifkan hanya classification score yang berubah sedangkan box tensor tetap.


In [ ]:
static_cmd = [
    sys.executable,'-u','-m','coffee_detector.experiments.run_af2_orient_cmc0',
    '--stage','static',
    '--data-root',str(DATA_ROOT),
    '--grouped-summary',str(GROUPED),
    '--d0-checkpoint',str(D0),
    '--output-root',str(OUTPUT),
    '--seed','42','--device','0',
]
print('MENJALANKAN:', ' '.join(static_cmd), flush=True)
subprocess.run(static_cmd, cwd=REPO, check=True)
STATIC = OUTPUT/'static_audit.json'
static_payload = json.loads(STATIC.read_text())
print(json.dumps(static_payload, indent=2))
assert static_payload['decision'] == 'PASS', static_payload


## 2. Train/resume seed42

Runner bersifat resume-safe. Jika `last.pt` masih resumable, rerun cell ini akan melanjutkan run yang sama. Candidate tidak dimulai dari generic `yolo26n.pt`; custom trainer mentransfer matched D0 seed42.


In [ ]:
train_cmd = [
    sys.executable,'-u','-m','coffee_detector.experiments.run_af2_orient_cmc0',
    '--stage','train',
    '--data-root',str(DATA_ROOT),
    '--grouped-summary',str(GROUPED),
    '--d0-checkpoint',str(D0),
    '--af2-orient-result',str(AF2_ORIENT_PARENT),
    '--output-root',str(OUTPUT),
    '--seed','42','--device','0','--latency-iterations','50',
    '--authorize-training',
]
print('MENJALANKAN:', ' '.join(train_cmd), flush=True)
subprocess.run(train_cmd, cwd=REPO, check=True)


In [ ]:
RESULT = OUTPUT/'val_reports/AF2_ORIENT_CMC0_seed42_result.json'
assert RESULT.is_file(), RESULT
payload = json.loads(RESULT.read_text())
print('=== AF2_ORIENT + CMC0 SEED42 ===')
print('candidate:', json.dumps(payload['candidate'], indent=2))
print('parent   :', json.dumps(payload['parent']['AF2_ORIENT'], indent=2))
print('delta pp :')
for key, value in payload['comparison']['deltas'].items():
    print(f'  {key}: {value*100:+.3f} pp')
print('route    :', payload['comparison']['route'])
print('decision :', payload['decision'])
print('next     :', payload['next_action'])
print('test accessed:', payload['test_images_accessed'])
print('RESULT:', RESULT)
